# Recording analysis

Private interview-review snapshot. Start with **mock mode**, which generates synthetic temperature and IMU arrays without hardware or ROS 2. For real bags, set `ANALYSIS_MODE=mode1` (recordings folder) or `mode2` (one bag), configure paths and use a matching ROS 2 environment.

This notebook provides per-topic rates/counts, plots, consecutive-change flags, experiment-documentation checks and message-structure checks. Outputs and real recordings are excluded. See the repository README for setup and review scope.


---

<div style="margin:32px 0 14px;padding:18px 22px;border-left:6px solid #2474a8;background:#edf4fb;border-radius:8px">
<h2 style="margin:0 0 8px">Section 0 — Analysis Configuration</h2>
<p style="margin:0"><b>The generated HTML is a read-only result.</b> To change the mode and rerun, open the original notebook in JupyterLab/VS Code or use one of the commands below.</p>
</div>

| Choice | Use it for | Required input | Command |
|---|---|---|---|
| **Mode 1 — recordings folder** | Standard analysis of one or many collected experiments | `recordings/` root + device YAML | `make analysis ANALYSIS_MODE=mode1 ANALYSIS_RECORDINGS_DIR=../../recordings` |
| **Mode 2 — one bag** | Inspect one explicit rosbag with manually configured topics | Path to a `bag/` folder | `make analysis ANALYSIS_MODE=mode2 BAG_PATH=../../recordings/<exp>/<device>/bag` |
| **Mock — generated data** | Test the notebook without rosbag, hardware, or ROS2 | Optional duration | `make analysis ANALYSIS_MODE=mock MOCK_DURATION_S=30` |

In a live JupyterLab/VS Code session, use the Mode selector and click **Apply settings**, then rerun Section 1 and the following sections.


In [ ]:
# ============================================================
# STEP 1 — Select mode
# ============================================================
# Options: 'mock'  |  'mode1'  |  'mode2'
#
#   'mock'  → No hardware. Fill [MOCK] block only. Ignore [MODE 1] and [MODE 2].
#   'mode1' → Normal workflow (yaml + recordings folder). Fill [MODE 1] block only.
#             Ignore [MOCK] and [MODE 2].
#   'mode2' → Direct path, manual topic list. Fill [MODE 2] block only.
#             Ignore [MOCK] and [MODE 1].
#
# Mode 1 & 2 read real .mcap recordings via rosbag2_py, which requires a
# sourced ROS2 environment (see Dependencies in README.md).

import os

MODE = os.environ.get('ANALYSIS_MODE', 'mock').strip().lower()
if MODE not in {'mode1', 'mode2', 'mock'}:
    raise ValueError(f'Invalid ANALYSIS_MODE={MODE!r}; choose mode1, mode2, or mock')


# ============================================================
# STEP 2 — Fill in ONLY the block that matches your MODE above.
#           Leave the other two blocks unchanged.
# ============================================================

# ------------------------------------------------------------
# [MOCK]  ← fill this block when MODE = 'mock'
#            ignore when MODE = 'mode1' or 'mode2'
#
# No hardware needed. Uses the same parameters as the mock
# publishers in rdc_hw/simulation/. Nothing else to configure.
# ------------------------------------------------------------
MOCK_DURATION_S = int(os.environ.get('MOCK_DURATION_S', '30'))  # simulated seconds


# ------------------------------------------------------------
# [MODE 1]  ← fill this block when MODE = 'mode1'
#              ignore when MODE = 'mock' or 'mode2'
#
# Reads which topics to analyze from the device yaml (analysis block),
# then auto-scans the recordings folder to find experiments.
# ------------------------------------------------------------

# Path to the device config yaml
DEVICE_CONFIG = os.environ.get('DEVICE_CONFIG', '../devices_config/device1.example.yaml')

# Root folder where all recordings are stored.
# Default: this project's recordings/ directory.
# Other users: replace this with a relative or absolute path that is accessible
# from the computer or Docker container running this notebook.
RECORDINGS_DIR = os.environ.get('ANALYSIS_RECORDINGS_DIR', '../../recordings').strip() or '../../recordings'

# Which experiments to include:
#   []                             → all experiments found in RECORDINGS_DIR
#   ['exp_20260707T142107Z_review_pc_59c1ca5b'] → one specific experiment
#   ['exp_001_...', 'exp_002_...'] → compare multiple experiments side by side
#
# Tip: when validating experiment_documentation.yaml, select the specific
# experiment you just ran. Otherwise old recordings without documentation
# templates may produce expected warnings.
# For CLI execution, this can also be set with:
#   EXPERIMENT_IDS=exp_001,exp_002 jupyter nbconvert --execute ...
_experiment_ids_env = os.environ.get('EXPERIMENT_IDS', '').strip()
EXPERIMENT_IDS = [x.strip() for x in _experiment_ids_env.split(',') if x.strip()] if _experiment_ids_env else []


# ------------------------------------------------------------
# [MODE 2]  ← fill this block when MODE = 'mode2'
#              ignore when MODE = 'mock' or 'mode1'
#
# Use this when you want to point directly at a specific bag folder,
# or analyze topics that are not listed in the device yaml.
# ------------------------------------------------------------

# Path to the bag FOLDER (contains metadata.yaml + bag_0.mcap) — not the .mcap file itself
# e.g. '../../recordings/exp_001_20260629T100000Z/device1_20260629T100001Z/bag'
BAG_PATH = os.environ.get('BAG_PATH', '').strip() or '../../recordings/<experiment>/<device>/bag'

# Topics to analyze — add/remove entries as needed
MANUAL_TOPICS = {
    '/temperature': {'expected_hz': 1.0,  'anomaly_threshold': 10.0},
    # '/imu/data':    {'expected_hz': 10.0, 'anomaly_threshold': 5.0},
    # '/camera/image_raw': {'expected_hz': 10.0},   # frame rate only, no value plot
}


# ============================================================
# STEP 3 — Plot settings  (apply to all modes)
# ============================================================
SAVE_PLOTS  = True    # Save plots as .png files
SHOW_ZOOMED = True    # Show a second zoomed-in subplot when anomalies are present

# Where plots are saved, mirroring the recordings/ experiment/device folder structure:
#   <OUTPUT_DIR>/<exp_id>/<device_dir>/<topic>.png      (Mode 1, Mode 2)
#   <OUTPUT_DIR>/mock_<duration>s/<topic>.png            (Mock)
OUTPUT_DIR = '../../recording_analysis_output'

# In a live JupyterLab/VS Code session, the path can also be changed without
# editing source code. Click Apply, then rerun Section 1 and the sections below.
HEADLESS_ANALYSIS = os.environ.get('HEADLESS_ANALYSIS') == '1'
if not HEADLESS_ANALYSIS:
    try:
        import ipywidgets as widgets
        from IPython.display import display
        mode_input = widgets.Dropdown(
            options=[('Mode 1 — recordings folder', 'mode1'), ('Mode 2 — one bag path', 'mode2'), ('Mock — generated data', 'mock')],
            value=MODE,
            description='Mode:',
            layout=widgets.Layout(width='75%'),
            style={'description_width': '130px'},
        )
        recordings_path_input = widgets.Text(
            value=RECORDINGS_DIR,
            description='Recordings path:',
            placeholder='Relative or absolute path',
            layout=widgets.Layout(width='75%'),
            style={'description_width': '130px'},
        )
        apply_recordings_path = widgets.Button(
            description='Apply settings', button_style='primary', icon='check')
        recordings_path_status = widgets.Output()

        def _update_path_state(change=None):
            recordings_path_input.disabled = mode_input.value != 'mode1'

        def _apply_recordings_path(_):
            global MODE, RECORDINGS_DIR
            candidate = os.path.expanduser(recordings_path_input.value.strip())
            with recordings_path_status:
                recordings_path_status.clear_output()
                if mode_input.value == 'mode1' and not candidate:
                    print('Please enter a recordings directory.')
                elif mode_input.value == 'mode1' and not os.path.isdir(candidate):
                    print(f'Path not found in this analysis environment: {candidate}')
                else:
                    MODE = mode_input.value
                    if MODE == 'mode1':
                        RECORDINGS_DIR = candidate
                        print(f'Applied Mode 1 path: {os.path.abspath(RECORDINGS_DIR)}')
                    else:
                        print(f'Applied mode: {MODE}')
                        if MODE == 'mode2':
                            print('Set BAG_PATH and MANUAL_TOPICS in this cell before continuing.')
                    print('Now rerun Section 1 and all following sections.')

        mode_input.observe(_update_path_state, names='value')
        _update_path_state()
        apply_recordings_path.on_click(_apply_recordings_path)
        display(widgets.VBox([
            widgets.HTML('<b>Interactive analysis input</b><br><small>Paths must be accessible from the machine or container running this notebook.</small>'),
            mode_input,
            widgets.HBox([recordings_path_input, apply_recordings_path]),
            recordings_path_status,
        ]))
    except ImportError:
        print('ipywidgets is not installed; edit RECORDINGS_DIR in this cell instead.')

# A compact, visible summary is retained in exported HTML even when code input
# is hidden by nbconvert. This separates effective settings from the user guide.
from html import escape
from IPython.display import display, HTML
if MODE == 'mode1':
    config_source = RECORDINGS_DIR
    config_scope = ', '.join(EXPERIMENT_IDS) if EXPERIMENT_IDS else 'All experiments found in recordings/'
elif MODE == 'mode2':
    config_source = BAG_PATH
    config_scope = ', '.join(MANUAL_TOPICS.keys()) or 'No topics selected'
else:
    config_source = 'Generated mock data'
    config_scope = f'{MOCK_DURATION_S} seconds'
display(HTML(f'''<div style="display:grid;grid-template-columns:repeat(3,minmax(180px,1fr));gap:10px;margin:10px 0 28px">
<div style="padding:12px;border:1px solid #d8dee8;border-radius:8px"><small>MODE</small><br><b>{escape(MODE)}</b></div>
<div style="padding:12px;border:1px solid #d8dee8;border-radius:8px"><small>DATA SOURCE</small><br><b>{escape(config_source)}</b></div>
<div style="padding:12px;border:1px solid #d8dee8;border-radius:8px"><small>EXPERIMENT / SCOPE</small><br><b>{escape(config_scope)}</b></div>
</div>'''))


---
## Section 1 — Load Data

In [ ]:
import os
import re
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# data structure:
# datasets = [
#   {
#     'label': 'exp_001 / device1',
#     'subdir': 'exp_001_.../device1_...',    # where Section 2 saves this dataset's plots
#     'bag_dir': '.../bag' or None (mock mode),   # used in Section 3 to cross-check metadata.yaml
#     'topics': {
#       '/temperature': {'timestamps': np.array, 'values': np.array or None,
#                        'expected_hz': float, 'anomaly_threshold': float or None}
#     }
#   }, ...
# ]
datasets = []

# Preserve physical units for supported scalar topics even when a bag contains
# zero messages. Timestamp-only camera streams are displayed as message counts.
TOPIC_DEFAULT_UNITS = {
    '/temperature': '°C',
    '/imu/data': 'm/s²',
    '/camera/image_raw': None,
}


def read_ros2_bag_topics(bag_dir, topics):
    """Yield (topic, ros_msg, timestamp_ns) for the given topics from a rosbag2
    bag folder (mcap storage), using the ROS2 rosbag2_py API. Requires a
    sourced ROS2 environment (rosbag2_py, rclpy, rosidl_runtime_py)."""
    import rosbag2_py
    from rclpy.serialization import deserialize_message
    from rosidl_runtime_py.utilities import get_message

    reader = rosbag2_py.SequentialReader()
    storage_options = rosbag2_py.StorageOptions(uri=bag_dir, storage_id='mcap')
    converter_options = rosbag2_py.ConverterOptions(
        input_serialization_format='cdr', output_serialization_format='cdr')
    reader.open(storage_options, converter_options)

    type_map = {t.name: t.type for t in reader.get_all_topics_and_types()}
    reader.set_filter(rosbag2_py.StorageFilter(topics=topics))

    while reader.has_next():
        topic, data, timestamp_ns = reader.read_next()
        msg = deserialize_message(data, get_message(type_map[topic]))
        yield topic, msg, timestamp_ns


def _device_id_from_dir(dev_dir):
    """'device1_20260629T100001Z' -> 'device1' (strips the trailing UTC timestamp)."""
    return re.sub(r'_\d{8}T\d{6}Z$', '', dev_dir)


# ----------------------------------------------------------
# MOCK MODE
# ----------------------------------------------------------
if MODE == 'mock':
    print('[Mock mode] Generating simulated data')
    print('  Parameters match mock publishers in rdc_hw/simulation/')
    random.seed(42)
    rng = np.random.default_rng(42)
    n = MOCK_DURATION_S

    topics = {}

    # /temperature — matches mock_temperature_publisher.py: gauss(25.0, 0.3) at 1 Hz
    ts = np.arange(0, n, 1.0)
    vals = np.array([random.gauss(25.0, 0.3) for _ in range(len(ts))])
    if len(vals) > 11:
        vals[10] = 60.0   # artificial spike — upper hardware limit (+60 °C)
        vals[11] = 25.2
    if len(vals) > 21:
        vals[20] = -40.0  # artificial spike — lower hardware limit (-40 °C)
        vals[21] = 24.8
    topics['/temperature'] = {'timestamps': ts, 'values': vals,
                               'expected_hz': 1.0, 'anomaly_threshold': 10.0,
                               'sensor_range': [-40, 60],
                               'unit': '°C'}

    # /imu/data — IAM-20685HP, plotting linear_acceleration.x
    # At rest: x ≈ 0, y ≈ 0, z ≈ 9.8 m/s² (gravity)
    # Default accel range: ±2g (±19.6 m/s²); anomaly threshold: 2.0 m/s²
    ts_imu = np.arange(0, n, 0.1)
    vals_imu = rng.normal(0.0, 0.05, len(ts_imu))  # x-axis ≈ 0 at rest, noise ~0.05 m/s²
    if len(vals_imu) > 51:
        vals_imu[50] = 12.0  # artificial spike
        vals_imu[51] = 0.05
    topics['/imu/data'] = {'timestamps': ts_imu, 'values': vals_imu,
                            'expected_hz': 10.0, 'anomaly_threshold': 2.0,
                            'sensor_range': None,
                            'unit': 'm/s²'}

    # /camera/image_raw — frame rate only, no values
    ts_cam = np.arange(0, n, 0.1)
    topics['/camera/image_raw'] = {'timestamps': ts_cam, 'values': None,
                                    'expected_hz': 10.0, 'anomaly_threshold': None,
                                    'unit': None}

    datasets.append({'label': f'mock ({n}s)', 'subdir': f'mock_{n}s', 'bag_dir': None, 'topics': topics})
    print(f'  Generated {n}s of data for {list(topics.keys())}')


# ----------------------------------------------------------
# MODE 1 — YAML-driven
# ----------------------------------------------------------
elif MODE == 'mode1':
    import yaml

    def _load_yaml(path):
        with open(path, 'r', encoding='utf-8') as f:
            return yaml.safe_load(f) or {}

    def _enabled_topics(cfg):
        return {t: v for t, v in cfg.get('analysis', {}).items() if v.get('enabled', True)}

    # Build device_name -> its own enabled-topics map from every yaml in the same folder
    # as DEVICE_CONFIG, so each Pi's topics come from ITS OWN config instead of forcing
    # every Pi in the experiment to share whichever single yaml DEVICE_CONFIG points to —
    # different Pis can (and do) record different topics/sensors.
    devices_config_dir = os.path.dirname(DEVICE_CONFIG) or '.'
    device_topics_map = {}
    for fname in os.listdir(devices_config_dir):
        if not fname.endswith(('.yaml', '.yml')):
            continue
        cfg_i = _load_yaml(os.path.join(devices_config_dir, fname))
        name = cfg_i.get('device_name')
        if name:
            device_topics_map[name] = _enabled_topics(cfg_i)

    # Fallback for a device_id with no matching yaml (e.g. simulated device IDs like
    # 'device1' that don't correspond to a devices_config/*.yaml file).
    fallback_topics = _enabled_topics(_load_yaml(DEVICE_CONFIG))

    print(f'[Mode 1] Device configs found in: {devices_config_dir}')
    print(f'  Devices: {[(name, list(t.keys())) for name, t in device_topics_map.items()]}')
    print(f'  Fallback ({os.path.basename(DEVICE_CONFIG)}): {list(fallback_topics.keys())}')

    # Scan recordings folder
    all_exps = sorted([d for d in os.listdir(RECORDINGS_DIR)
                       if os.path.isdir(os.path.join(RECORDINGS_DIR, d))])
    selected = EXPERIMENT_IDS if EXPERIMENT_IDS else all_exps
    print(f'  Experiments found: {all_exps}')
    print(f'  Selected: {selected}')

    for exp_id in selected:
        exp_dir = os.path.join(RECORDINGS_DIR, exp_id)
        device_dirs = [d for d in os.listdir(exp_dir) if os.path.isdir(os.path.join(exp_dir, d))]
        for dev_dir in device_dirs:
            bag_dir = os.path.join(exp_dir, dev_dir, 'bag')
            if not os.path.exists(os.path.join(bag_dir, 'metadata.yaml')):
                print(f'  [skip] {bag_dir} not found')
                continue

            device_id = _device_id_from_dir(dev_dir)
            if device_id in device_topics_map:
                active_topics = device_topics_map[device_id]
            else:
                print(f'  [{device_id}] no matching yaml in {devices_config_dir}, using fallback topics')
                active_topics = fallback_topics

            print(f'  Reading: {exp_id}/{dev_dir}  (topics: {list(active_topics.keys())})')
            topics = {t: {'timestamps': [], 'values': [],
                           'expected_hz': v.get('expected_hz'),
                           'anomaly_threshold': v.get('anomaly_threshold'),
                           'sensor_range': v.get('sensor_range'),
                           'unit': ''}
                      for t, v in active_topics.items()}
            t0_map = {}

            for topic, ros_msg, timestamp_ns in read_ros2_bag_topics(bag_dir, list(active_topics.keys())):
                t_s = timestamp_ns / 1e9
                if topic not in t0_map:
                    t0_map[topic] = t_s
                topics[topic]['timestamps'].append(t_s - t0_map[topic])
                if topic == '/temperature':
                    topics[topic]['values'].append(ros_msg.temperature)
                    topics[topic]['unit'] = '°C'
                elif topic == '/imu/data':
                    topics[topic]['values'].append(ros_msg.linear_acceleration.x)
                    topics[topic]['unit'] = 'm/s²'
                elif topic == '/camera/image_raw':
                    topics[topic]['unit'] = None

            for t in topics:
                topics[t]['timestamps'] = np.array(topics[t]['timestamps'])
                topics[t]['values'] = np.array(topics[t]['values']) if topics[t]['values'] else None

            datasets.append({'label': f'{exp_id} / {dev_dir}', 'subdir': os.path.join(exp_id, dev_dir),
                              'bag_dir': bag_dir, 'topics': topics})


# ----------------------------------------------------------
# MODE 2 — Manual
# ----------------------------------------------------------
elif MODE == 'mode2':
    assert BAG_PATH, 'Please set BAG_PATH in Section 0'

    print(f'[Mode 2] Reading: {BAG_PATH}')
    topics = {t: {'timestamps': [], 'values': [],
                   'expected_hz': v.get('expected_hz'),
                   'anomaly_threshold': v.get('anomaly_threshold'),
                   'sensor_range': v.get('sensor_range'),
                   'unit': ''}
              for t, v in MANUAL_TOPICS.items()}
    t0_map = {}

    for topic, ros_msg, timestamp_ns in read_ros2_bag_topics(BAG_PATH, list(MANUAL_TOPICS.keys())):
        t_s = timestamp_ns / 1e9
        if topic not in t0_map:
            t0_map[topic] = t_s
        topics[topic]['timestamps'].append(t_s - t0_map[topic])
        if topic == '/temperature':
            topics[topic]['values'].append(ros_msg.temperature)
            topics[topic]['unit'] = '°C'
        elif topic == '/imu/data':
            topics[topic]['values'].append(ros_msg.linear_acceleration.x)
            topics[topic]['unit'] = 'm/s²'
        elif topic == '/camera/image_raw':
            topics[topic]['unit'] = None

    for t in topics:
        topics[t]['timestamps'] = np.array(topics[t]['timestamps'])
        topics[t]['values'] = np.array(topics[t]['values']) if topics[t]['values'] else None

    # Mirror Mode 1's subdir scheme (<exp_id>/<device_dir>) so the same recording
    # produces the same output path regardless of which mode was used to read it.
    dev_dir_name = os.path.basename(os.path.dirname(BAG_PATH))
    exp_id_name = os.path.basename(os.path.dirname(os.path.dirname(BAG_PATH)))
    datasets.append({'label': dev_dir_name, 'subdir': os.path.join(exp_id_name, dev_dir_name),
                      'bag_dir': BAG_PATH, 'topics': topics})


# Summary
loaded_experiments = {os.path.normpath(ds['subdir']).split(os.sep)[0] for ds in datasets}
print(f'\nLoaded {len(loaded_experiments)} experiment(s), {len(datasets)} device recording(s):')
for ds in datasets:
    print(f'  [{ds["label"]}]')
    for t, d in ds['topics'].items():
        n = len(d['timestamps'])
        dur = d['timestamps'][-1] if n > 0 else 0
        print(f'    {t}: {n} messages, {dur:.1f}s')

---
## Section 2 — Plots

In [ ]:
print('=== Section 2 — Plots ===\n')

for ds in datasets:
    label = ds['label']
    out_dir = os.path.join(OUTPUT_DIR, ds['subdir'])
    if SAVE_PLOTS:
        os.makedirs(out_dir, exist_ok=True)

    for topic, d in ds['topics'].items():
        values = d['values']
        if values is None:
            continue  # timestamps-only topics (e.g. /camera/image_raw) have no value plot

        ts = d['timestamps']
        threshold = d.get('anomaly_threshold')
        sensor_range = d.get('sensor_range')
        unit = d.get('unit') or ''

        # anomaly detection: consecutive-difference threshold
        if threshold is not None and len(values) > 1:
            diffs = np.abs(np.diff(values))
            anomaly_idx = np.where(diffs > threshold)[0] + 1
        else:
            anomaly_idx = np.array([], dtype=int)

        has_anomalies = len(anomaly_idx) > 0
        show_zoom = SHOW_ZOOMED  # always show the zoomed subplot, regardless of anomalies

        fig, axes = plt.subplots(2 if show_zoom else 1, 1,
                                  figsize=(10, 8 if show_zoom else 4))
        axes = np.atleast_1d(axes)

        # --- top subplot: full range ---
        ax = axes[0]
        ax.plot(ts, values, '-o', markersize=3, label=topic)
        if has_anomalies:
            ax.scatter(ts[anomaly_idx], values[anomaly_idx], color='red', zorder=5, label='anomaly')
        if sensor_range is not None:
            ax.set_ylim(sensor_range)
        ax.set_xlabel('time (s)')
        ax.set_ylabel(f'{topic} ({unit})' if unit else topic)
        ax.set_title(f'{label} — {topic} (full range)')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)

        # --- bottom subplot: zoomed to the actual data range (mean +/- 3 sigma) ---
        if show_zoom:
            ax2 = axes[1]
            mean, std = np.mean(values), np.std(values)
            ax2.plot(ts, values, '-o', markersize=3)
            if has_anomalies:
                ax2.scatter(ts[anomaly_idx], values[anomaly_idx], color='red', zorder=5)
            if std > 0:
                ax2.set_ylim(mean - 3 * std, mean + 3 * std)
            ax2.set_xlabel('time (s)')
            ax2.set_ylabel(f'{topic} ({unit})' if unit else topic)
            ax2.set_title(f'{label} — {topic} (zoomed to actual range)')
            ax2.grid(True, alpha=0.3)

        plt.tight_layout()

        if SAVE_PLOTS:
            fname = os.path.join(out_dir, f"{topic.strip('/').replace('/', '_')}.png")
            plt.savefig(fname, dpi=100)
            print(f'Saved: {fname}')

        plt.show()


## Section 2b — Interactive Analysis Dashboard

Use the left-side controls in order: searchable **Experiment ID**, **Device**, then **Topic**. The right side shows full-range and detailed plots together with status, messages, duration, actual/expected frequency, anomalies, and unit. Plotly provides wheel/box zoom, hover, autoscale, axis reset, drawing tools, and PNG export.

In [ ]:
print('=== Section 2b — Interactive Analysis Dashboard ===\n')
from IPython.display import HTML
from plotly.offline import get_plotlyjs
import json

# Camera and other timestamp-only topics use cumulative message count.
dashboard_records=[]
for ds in datasets:
    parts=os.path.normpath(ds['subdir']).split(os.sep); experiment=parts[0]
    device=re.sub(r'_\d{8}T\d{6}Z$','',parts[1] if len(parts)>1 else parts[0])
    for topic,data in ds['topics'].items():
        x=np.asarray(data['timestamps'],dtype=float); raw=data.get('values')
        # Only timestamp-only camera streams are count series. An empty scalar
        # sensor (for example /temperature with 0 messages) remains a sensor series.
        count_mode=(topic == '/camera/image_raw')
        y=np.arange(1,len(x)+1,dtype=float) if count_mode else np.asarray(raw if raw is not None else [],dtype=float)
        unit='messages' if count_mode else (data.get('unit') or TOPIC_DEFAULT_UNITS.get(topic) or 'value')
        duration=float(x[-1]-x[0]) if len(x)>1 else 0.0; actual=float((len(x)-1)/duration) if duration>0 else 0.0
        expected=data.get('expected_hz'); deviation=abs(actual-expected)/expected*100 if expected and duration>0 else None
        threshold=data.get('anomaly_threshold'); anomalies=int(np.sum(np.abs(np.diff(y))>threshold)) if not count_mode and threshold is not None and len(y)>1 else 0
        status='NO DATA' if len(x)==0 else ('CHECK' if deviation is not None and deviation>10 else 'PASS')
        dashboard_records.append({'experiment':experiment,'device':device,'topic':topic,'x':x.tolist(),'y':y.tolist(),'unit':unit,
          'series_kind':'Cumulative messages' if count_mode else 'Sensor value','messages':len(x),'duration':round(duration,3),
          'expected_hz':expected,'actual_hz':round(actual,3),'frequency_deviation':None if deviation is None else round(deviation,2),
          'anomalies':anomalies,'status':status,'sensor_range':data.get('sensor_range')})

if not dashboard_records:
    print('No datasets are available for the interactive dashboard.')
else:
    records_json=json.dumps(dashboard_records,ensure_ascii=False).replace('</','<\\/')
    template=r'''<style>
#rdc-dashboard{font-family:Inter,Segoe UI,Arial,sans-serif;border:1px solid #d8dee8;border-radius:14px;overflow:hidden;background:#f6f8fb;margin:12px 0 28px;box-shadow:0 6px 24px rgba(30,55,90,.08)}#rdc-dashboard *{box-sizing:border-box}.rdc-head{background:linear-gradient(135deg,#173b65,#2474a8);color:#fff;padding:18px 22px}.rdc-head h2{margin:0 0 5px}.rdc-head p{margin:0;opacity:.88;font-size:13px}.rdc-body{display:grid;grid-template-columns:275px minmax(0,1fr);min-height:735px}.rdc-side{background:#fff;border-right:1px solid #d8dee8;padding:18px;display:flex;flex-direction:column;gap:15px}.rdc-side label{display:block;font-size:12px;font-weight:700;color:#405064;margin-bottom:5px}.rdc-side input,.rdc-side select{width:100%;padding:10px;border:1px solid #b8c3d1;border-radius:8px;background:#fff}.rdc-hint{font-size:11px;color:#69798b;line-height:1.5}.rdc-note{margin-top:auto;padding:10px;background:#edf4fb;border-radius:8px;font-size:11px;color:#41576e;line-height:1.45}.rdc-main{padding:16px;min-width:0}.rdc-cards{display:grid;grid-template-columns:repeat(6,minmax(100px,1fr));gap:8px;margin-bottom:10px}.rdc-card{background:#fff;border:1px solid #dce3ec;border-radius:9px;padding:10px}.rdc-card small{display:block;color:#6c798a;font-size:10px;text-transform:uppercase}.rdc-card strong{display:block;color:#17283e;font-size:16px;margin-top:4px;overflow-wrap:anywhere}.PASS{color:#087f5b!important}.CHECK,.NO-DATA{color:#c05a00!important}.rdc-plot{height:610px;background:#fff;border:1px solid #dce3ec;border-radius:10px}@media(max-width:950px){.rdc-body{grid-template-columns:1fr}.rdc-side{border-right:0;border-bottom:1px solid #ddd;display:grid;grid-template-columns:repeat(3,1fr)}.rdc-note{grid-column:1/-1}.rdc-cards{grid-template-columns:repeat(3,1fr)}}
</style><script>__PLOTLY__</script>
<div id="rdc-dashboard"><div class="rdc-head"><h2>Recording Analysis Dashboard</h2><p>Experiment → Device → Topic · full range + detailed view · zoom, hover and reset</p></div><div class="rdc-body"><aside class="rdc-side">
<div><label>1. Experiment ID (type or select)</label><input id="rdc-exp" list="rdc-exp-list" autocomplete="off"><datalist id="rdc-exp-list"></datalist></div>
<div><label>2. Device</label><select id="rdc-device"></select></div><div><label>3. Topic</label><select id="rdc-topic"></select></div>
<div class="rdc-hint">Mouse wheel: zoom. Drag: box zoom. Double-click: reset. The toolbar also provides zoom in/out, autoscale, reset and PNG download.</div>
<div class="rdc-note"><b>Documentation:</b> this generated HTML is read-only. Open the original notebook in JupyterLab/VS Code to fill and save YAML fields.</div></aside><main class="rdc-main">
<div class="rdc-cards"><div class="rdc-card"><small>Status</small><strong id="rdc-status"></strong></div><div class="rdc-card"><small>Messages</small><strong id="rdc-messages"></strong></div><div class="rdc-card"><small>Duration</small><strong id="rdc-duration"></strong></div><div class="rdc-card"><small>Actual / expected</small><strong id="rdc-frequency"></strong></div><div class="rdc-card"><small>Anomalies</small><strong id="rdc-anomalies"></strong></div><div class="rdc-card"><small>Unit</small><strong id="rdc-unit"></strong></div></div><div id="rdc-plot" class="rdc-plot"></div></main></div></div>
<script>(()=>{const R=__DATA__,E=document.getElementById('rdc-exp'),L=document.getElementById('rdc-exp-list'),D=document.getElementById('rdc-device'),T=document.getElementById('rdc-topic'),U=a=>[...new Set(a)],O=(e,v)=>e.innerHTML=v.map(x=>`<option value="${x}">${x}</option>`).join('');const EX=U(R.map(x=>x.experiment));L.innerHTML=EX.map(x=>`<option value="${x}"></option>`).join('');E.value=EX[0];
function exact(){return EX.includes(E.value)?E.value:EX.find(x=>x.toLowerCase().includes(E.value.toLowerCase()))||EX[0]}function devices(){E.value=exact();O(D,U(R.filter(x=>x.experiment===E.value).map(x=>x.device)));topics()}function topics(){O(T,U(R.filter(x=>x.experiment===E.value&&x.device===D.value).map(x=>x.topic)));draw()}function f(v){return v==null?'—':v}
function draw(){const r=R.find(x=>x.experiment===E.value&&x.device===D.value&&x.topic===T.value);if(!r)return;const m=r.y.length?r.y.reduce((a,b)=>a+b,0)/r.y.length:0,s=Math.sqrt(r.y.length?r.y.reduce((a,b)=>a+(b-m)**2,0)/r.y.length:0),lo=s?m-3*s:m-1,hi=s?m+3*s:m+1;document.getElementById('rdc-status').innerHTML=`<span class="${r.status.replace(' ','-')}">${r.status==='PASS'?'✓ ':r.status==='CHECK'?'⚠ ':'— '}${r.status}</span>`;document.getElementById('rdc-messages').textContent=r.messages;document.getElementById('rdc-duration').textContent=r.duration.toFixed(2)+' s';document.getElementById('rdc-frequency').textContent=`${r.actual_hz} / ${f(r.expected_hz)} Hz`;document.getElementById('rdc-anomalies').textContent=r.anomalies;document.getElementById('rdc-unit').textContent=r.unit;
const tr={x:r.x,y:r.y,type:'scatter',mode:'lines+markers',marker:{size:5,color:'#2474a8'},line:{width:2,color:'#2474a8'},hovertemplate:`time=%{x:.3f} s<br>value=%{y:.4f} ${r.unit}<extra>${r.device}</extra>`},full=r.sensor_range&&r.sensor_range.length===2?r.sensor_range:undefined,layout={grid:{rows:2,columns:1,pattern:'independent'},height:600,margin:{l:78,r:25,t:65,b:55},showlegend:false,hovermode:'x unified',paper_bgcolor:'#fff',plot_bgcolor:'#fbfcfe',title:{text:`${r.experiment}<br><sup>${r.device} · ${r.topic} · ${r.series_kind}</sup>`,x:.02},xaxis:{title:'Time (s)'},yaxis:{title:`Full range (${r.unit})`,range:full,domain:[.56,1]},xaxis2:{title:'Time (s)'},yaxis2:{title:`Detailed view (${r.unit})`,range:[lo,hi],domain:[0,.42]},annotations:[{text:'FULL RANGE',xref:'paper',yref:'paper',x:0,y:1.04,showarrow:false},{text:'ZOOMED / DETAIL',xref:'paper',yref:'paper',x:0,y:.45,showarrow:false}]};Plotly.react('rdc-plot',[tr,{...tr,xaxis:'x2',yaxis:'y2'}],layout,{responsive:true,scrollZoom:true,displaylogo:false,modeBarButtonsToAdd:['drawline','eraseshape'],toImageButtonOptions:{format:'png',scale:2}})}E.onchange=devices;E.onkeyup=e=>{if(e.key==='Enter')devices()};D.onchange=topics;T.onchange=draw;devices()})();</script>'''
    display(HTML(template.replace('__PLOTLY__',get_plotlyjs()).replace('__DATA__',records_json)))
    print(f'Interactive dashboard: {len(set(r["experiment"] for r in dashboard_records))} experiment(s), {len(set(r["device"] for r in dashboard_records))} device(s), {len(dashboard_records)} series.')


---
## Section 3a — Experiment Documentation Check

Select one, several, or all experiments to review required documentation fields. Generated HTML is read-only; use the live JupyterLab/VS Code form to edit and save YAML.


In [ ]:
print('=== Section 3a — Experiment Documentation Check ===\\n')
import pandas as pd
from html import escape
from IPython.display import display, HTML
MANDATORY_DOC_FIELDS=['operator','goal','material_type','measurement_series','setup_notes']
DOC_LABELS={'operator':'Operator','goal':'Goal','material_type':'Material type','measurement_series':'Measurement series','setup_notes':'Setup notes'}

def _experiment_ids_from_datasets(datasets):
    ids=[]
    for ds in datasets:
        subdir=ds.get('subdir') or ''
        if not subdir.startswith('mock_'):
            exp_id=subdir.split(os.sep)[0].split('/')[0]
            if exp_id and exp_id not in ids: ids.append(exp_id)
    return ids

def _load_experiment_doc(path):
    try: import yaml
    except ImportError: return None
    if not os.path.exists(path): return None
    with open(path,encoding='utf-8') as handle: return yaml.safe_load(handle) or {}

def _filterable_experiment_html(prefix, sections, table_headers=None):
    if not sections: return '<p>No experiments are available.</p>'
    choices=''.join(f'<label><input type="checkbox" value="{escape(exp)}" {"checked" if i==len(sections)-1 else ""}> <span>{escape(exp)}</span></label>' for i,(exp,_) in enumerate(sections))
    if table_headers:
        header=''.join(f'<th>{escape(column)}</th>' for column in table_headers)
        rows=''.join(f'<tr class="rdc-exp-section" data-display="table-row" data-experiment="{escape(exp)}" style="display:{"table-row" if i==len(sections)-1 else "none"}">{body}</tr>' for i,(exp,body) in enumerate(sections))
        bodies=f'<div class="rdc-table-wrap"><table class="rdc-summary-table"><thead><tr>{header}</tr></thead><tbody>{rows}</tbody></table></div>'
    else:
        bodies=''.join(f'<section class="rdc-exp-section" data-display="block" data-experiment="{escape(exp)}" style="display:{"block" if i==len(sections)-1 else "none"}">{body}</section>' for i,(exp,body) in enumerate(sections))
    return f'''<div id="{prefix}" class="rdc-exp-filter"><style>
#{prefix}{{border:1px solid #d8dee8;border-radius:10px;background:#fff;padding:14px;margin:10px 0 24px}}
#{prefix} .rdc-filter-bar{{display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin-bottom:12px}}
#{prefix} button{{border:1px solid #aeb9c7;border-radius:7px;background:#fff;padding:7px 11px;cursor:pointer}}
#{prefix} .rdc-filter-toggle{{background:#2474a8;color:#fff;border-color:#2474a8;min-width:190px;text-align:left}}
#{prefix} .rdc-filter-panel{{display:none;border:1px solid #ccd5e0;border-radius:8px;padding:10px;max-height:220px;overflow:auto;margin-bottom:14px;background:#f8fafc}}
#{prefix} .rdc-filter-panel.open{{display:grid;grid-template-columns:repeat(auto-fit,minmax(290px,1fr));gap:7px}}
#{prefix} .rdc-filter-panel label{{display:flex;align-items:flex-start;gap:7px;font-family:monospace;font-size:12px;overflow-wrap:anywhere}}
#{prefix} .rdc-empty{{display:none;padding:14px;background:#fff3cd;border-radius:7px;color:#7a4b00}}
#{prefix} section.rdc-exp-section{{border-top:3px solid #2474a8;padding-top:12px;margin-top:12px}}
#{prefix} .rdc-table-wrap{{overflow-x:auto}} #{prefix} .rdc-summary-table{{width:100%;border-collapse:collapse;font-size:13px}}
#{prefix} .rdc-summary-table th{{background:#edf4fb;text-align:left;padding:9px;border-bottom:2px solid #9fb8d1;white-space:nowrap}}
#{prefix} .rdc-summary-table td{{padding:9px;border-bottom:1px solid #dce3ec;vertical-align:top}} #{prefix} .rdc-summary-table tbody tr:hover{{background:#f6f9fc}}
#{prefix} .rdc-exp-name{{font-family:monospace;font-weight:600;overflow-wrap:anywhere}} #{prefix} .rdc-status{{display:inline-block;border-radius:999px;padding:3px 8px;font-weight:700;white-space:nowrap}}
</style><div class="rdc-filter-bar"><b>Experiments:</b><button class="rdc-filter-toggle">Selected: 1 ▾</button><button class="rdc-all">All</button><button class="rdc-clear">Clear</button></div>
<div class="rdc-filter-panel">{choices}</div><div class="rdc-empty">No experiment selected.</div>{bodies}
<script>(()=>{{const root=document.getElementById('{prefix}'),panel=root.querySelector('.rdc-filter-panel'),toggle=root.querySelector('.rdc-filter-toggle'),boxes=[...panel.querySelectorAll('input')],sections=[...root.querySelectorAll('.rdc-exp-section')];function apply(){{const selected=new Set(boxes.filter(x=>x.checked).map(x=>x.value));sections.forEach(s=>s.style.display=selected.has(s.dataset.experiment)?(s.dataset.display||'block'):'none');root.querySelector('.rdc-empty').style.display=selected.size?'none':'block';toggle.textContent=selected.size===boxes.length?'All experiments ▾':`Selected: ${{selected.size}} ▾`;setTimeout(()=>window.Plotly&&sections.filter(s=>s.style.display!=='none').forEach(s=>s.querySelectorAll('.js-plotly-plot').forEach(p=>Plotly.Plots.resize(p))),30)}}toggle.onclick=()=>panel.classList.toggle('open');boxes.forEach(x=>x.onchange=apply);root.querySelector('.rdc-all').onclick=()=>{{boxes.forEach(x=>x.checked=true);apply()}};root.querySelector('.rdc-clear').onclick=()=>{{boxes.forEach(x=>x.checked=false);apply()}};apply()}})();</script></div>'''

if MODE=='mock':
    doc_exp_ids=[];doc_status_rows=[];print('Mock mode: no experiment documentation file expected.')
else:
    doc_exp_ids=list(selected) if MODE=='mode1' else _experiment_ids_from_datasets(datasets)
    doc_status_rows=[];documentation_sections=[]
    for exp_id in doc_exp_ids:
        path=os.path.normpath(os.path.join(RECORDINGS_DIR,exp_id,'experiment_documentation.yaml'))
        doc=_load_experiment_doc(path);mandatory=(doc or {}).get('mandatory') or {}
        missing=MANDATORY_DOC_FIELDS if doc is None else [f for f in MANDATORY_DOC_FIELDS if not str(mandatory.get(f,'')).strip()]
        row={'Experiment':exp_id,'Status':'⚠ CHECK' if missing else '✅ COMPLETE','Completed':f'{5-len(missing)}/5','Missing fields':', '.join(DOC_LABELS[f] for f in missing) or '—','YAML path':path}
        doc_status_rows.append(row);color='#087f5b' if not missing else '#9a5600';bg='#dcfce7' if not missing else '#fff3cd'
        body=f'''<td class="rdc-exp-name">{escape(exp_id)}</td><td><span class="rdc-status" style="background:{bg};color:{color}">{row['Status']}</span></td><td style="text-align:center;font-weight:700">{row['Completed']}</td><td>{escape(row['Missing fields'])}</td><td><details><summary style="cursor:pointer;color:#2474a8">Show path</summary><code style="overflow-wrap:anywhere">{escape(path)}</code></details></td>'''
        documentation_sections.append((exp_id,body))
    display(HTML(_filterable_experiment_html('rdc-doc-filter',documentation_sections,['Experiment','Status','Completed','Missing fields','YAML'])))
    if os.environ.get('HEADLESS_ANALYSIS')=='1':
        display(HTML('<div style="padding:10px 14px;background:#edf4fb;border-left:4px solid #2474a8;border-radius:6px"><b>Read-only HTML.</b> Open the original notebook in JupyterLab or VS Code to edit and save mandatory YAML fields.</div>'))
    else:
        try:
            import yaml,ipywidgets as widgets
            exp_w=widgets.Dropdown(options=doc_exp_ids,description='Experiment:',layout=widgets.Layout(width='95%'))
            fields={f:widgets.Textarea(description=DOC_LABELS[f]+':',layout=widgets.Layout(width='95%',height='58px')) for f in MANDATORY_DOC_FIELDS}
            button=widgets.Button(description='Save mandatory fields to YAML',button_style='success',icon='save');status=widgets.HTML()
            def load(change=None):
                path=os.path.normpath(os.path.join(RECORDINGS_DIR,exp_w.value,'experiment_documentation.yaml'));doc=_load_experiment_doc(path) or {};mandatory=doc.get('mandatory') or {}
                for f,w in fields.items(): w.value=str(mandatory.get(f,'') or '')
                status.value=f'Editing: <code>{path}</code>'
            def save(_):
                path=os.path.normpath(os.path.join(RECORDINGS_DIR,exp_w.value,'experiment_documentation.yaml'));doc=_load_experiment_doc(path) or {'experiment_id':exp_w.value};doc['mandatory']={f:w.value.strip() for f,w in fields.items()};os.makedirs(os.path.dirname(path),exist_ok=True)
                with open(path,'w',encoding='utf-8') as handle: yaml.safe_dump(doc,handle,sort_keys=False,allow_unicode=True)
                missing=[DOC_LABELS[f] for f,w in fields.items() if not w.value.strip()];status.value='<b style="color:#087f5b">Saved.</b>' if not missing else '<b style="color:#c05a00">Saved; missing: '+', '.join(missing)+'</b>'
            exp_w.observe(load,names='value');button.on_click(save);load();display(widgets.VBox([widgets.HTML('<h4>Edit mandatory experiment documentation</h4>'),exp_w,*fields.values(),button,status]))
        except ImportError: print('ipywidgets/PyYAML unavailable; documentation form disabled.')


---
## Section 3 — Statistics

In [ ]:
import re
import pandas as pd
from IPython.display import display

print('=== Section 3 — Statistics ===\n')

DEVIATION_WARN_PCT = 10.0  # flag if count/frequency deviates more than this from expected

# Accumulated across all datasets below, in addition to the per-device tables printed here —
# reused by Section 3b (Experiment Overview) to build the per-experiment tables.
overview_rows = []


def _experiment_and_device(subdir):
    """'exp_001_.../device1_20260629T100001Z' -> ('exp_001_...', 'device1')."""
    parts = os.path.normpath(subdir).split(os.sep)
    exp_id = parts[0]
    device_id = re.sub(r'_\d{8}T\d{6}Z$', '', parts[1]) if len(parts) > 1 else parts[0]
    return exp_id, device_id


for ds in datasets:
    label = ds['label']
    bag_dir = ds.get('bag_dir')
    exp_id, device_id = _experiment_and_device(ds['subdir'])
    # Parent of bag/ — the actual per-Pi recording folder (bag, experiment_info.json,
    # metadata.json, logs.txt all live here). None in Mock mode, which has no real folder.
    directory = os.path.dirname(bag_dir) if bag_dir else None

    # Load actual recorded message counts from metadata.yaml (ground truth written
    # by rosbag2 at recording time), to cross-check against what we read ourselves.
    recorded_map = {}
    if bag_dir:
        meta_path = os.path.join(bag_dir, 'metadata.yaml')
        if os.path.exists(meta_path):
            import yaml
            with open(meta_path, 'r') as f:
                meta = yaml.safe_load(f)
            for entry in meta['rosbag2_bagfile_information']['topics_with_message_count']:
                recorded_map[entry['topic_metadata']['name']] = entry['message_count']

    rows = []
    anomaly_details = []

    for topic, d in ds['topics'].items():
        ts = d['timestamps']
        values = d['values']
        expected_hz = d.get('expected_hz')
        threshold = d.get('anomaly_threshold')
        unit = d.get('unit') or ''

        n = len(ts)
        duration = (ts[-1] - ts[0]) if n > 0 else 0.0
        actual_hz = (n - 1) / duration if duration > 0 else 0.0

        row = {'Topic': topic, 'Messages': n, 'Duration (s)': round(duration, 1)}

        if expected_hz:
            expected_count = expected_hz * duration
            count_dev = abs(n - expected_count) / expected_count * 100 if expected_count > 0 else 0.0
            hz_dev = abs(actual_hz - expected_hz) / expected_hz * 100 if expected_hz > 0 else 0.0
            row['Expected Hz'] = expected_hz
            row['Actual Hz'] = round(actual_hz, 3)
            row['Count dev'] = f"{'✅' if count_dev <= DEVIATION_WARN_PCT else '⚠️'} {count_dev:.1f}%"
            row['Freq dev'] = f"{'✅' if hz_dev <= DEVIATION_WARN_PCT else '⚠️'} {hz_dev:.1f}%"
        else:
            row['Expected Hz'] = row['Actual Hz'] = row['Count dev'] = row['Freq dev'] = '—'

        # Cross-check against metadata.yaml: catches bugs in OUR reading code,
        # independent of whether the sensor itself published at the right rate.
        recorded = recorded_map.get(topic)
        if recorded is not None:
            rec_dev = abs(n - recorded) / recorded * 100 if recorded > 0 else 0.0
            row['Recorded'] = recorded
            row['Recorded dev'] = f"{'✅' if rec_dev <= DEVIATION_WARN_PCT else '⚠️'} {rec_dev:.1f}%"
        else:
            row['Recorded'] = row['Recorded dev'] = '—'

        if values is not None and threshold is not None and n > 1:
            diffs = np.abs(np.diff(values))
            anomaly_idx = np.where(diffs > threshold)[0] + 1
            row['Anomalies'] = f"⚠️ {len(anomaly_idx)}" if len(anomaly_idx) > 0 else '✅ 0'
            for idx in anomaly_idx:
                anomaly_details.append(
                    f"  [{topic}] t={ts[idx]:.1f}s  value={values[idx]:.3f}  "
                    f"(prev={values[idx-1]:.3f}, Δ={diffs[idx-1]:.3f} {unit})")
        else:
            row['Anomalies'] = '—'

        if values is not None:
            row['Mean'] = round(float(np.mean(values)), 3)
            row['Std'] = round(float(np.std(values)), 4)
            row['Min'] = round(float(np.min(values)), 3)
            row['Max'] = round(float(np.max(values)), 3)
            row['Unit'] = unit
        else:
            row['Mean'] = row['Std'] = row['Min'] = row['Max'] = row['Unit'] = '—'

        check_values = [row.get('Count dev'), row.get('Freq dev'), row.get('Recorded dev'), row.get('Anomalies')]
        row['Status'] = ('— NO DATA' if n == 0 else
                         '⚠ CHECK' if any(str(value).startswith('⚠') for value in check_values) else '✅ PASS')
        rows.append(row)
        overview_rows.append({'Experiment': exp_id, 'Device': device_id, 'Directory': directory, **row})

    print(f'--- {label} ---')
    stats_df = pd.DataFrame(rows)
    ordered = ['Status', 'Topic', 'Messages', 'Duration (s)', 'Expected Hz', 'Actual Hz',
               'Count dev', 'Freq dev', 'Recorded', 'Recorded dev', 'Anomalies',
               'Mean', 'Std', 'Min', 'Max', 'Unit']
    stats_df = stats_df[[column for column in ordered if column in stats_df.columns]]
    def _quality_style(value):
        text = str(value)
        if text.startswith('✅'): return 'background-color:#dcfce7;color:#087f5b;font-weight:700'
        if text.startswith('⚠'): return 'background-color:#fff3cd;color:#9a5600;font-weight:700'
        if 'NO DATA' in text: return 'background-color:#f1f3f5;color:#68727d;font-weight:700'
        return ''
    display(stats_df.style.map(_quality_style, subset=[column for column in ['Status','Count dev','Freq dev','Recorded dev','Anomalies'] if column in stats_df.columns])
            .set_caption(f'Device statistics — {label}').set_properties(**{'padding':'7px','text-align':'right'})
            .set_properties(subset=['Topic'], **{'text-align':'left','font-weight':'600'}).hide(axis='index'))

    if anomaly_details:
        print('Anomaly details:')
        for line in anomaly_details:
            print(line)
    print()


---
## Section 3b — Experiment Overview

Select one, several, or all experiments. Each selected experiment shows its summary, device index, topic details, and interactive message-coverage chart.


In [ ]:
print('=== Section 3b — Experiment Overview ===\\n')
INDEX_COLS=['Device','IP','Directory','Topics']
DETAIL_COLS=['Status','Device','IP','Topic','Messages','Duration (s)','Mean','Std','Min','Max','Unit']

def _load_device_ips(devices_config_dir):
    ip_map={}
    try: import yaml
    except ImportError: return ip_map
    try:
        for filename in os.listdir(devices_config_dir):
            if not filename.endswith(('.yaml','.yml')): continue
            with open(os.path.join(devices_config_dir,filename),encoding='utf-8') as handle: config=yaml.safe_load(handle) or {}
            name,ip=config.get('device_name'),config.get('pi_ip')
            if name and ip: ip_map[name]=ip
    except (FileNotFoundError,NotADirectoryError): pass
    return ip_map

def _overview_status(value):
    text=str(value)
    if text.startswith('✅'): return 'background:#dcfce7;color:#087f5b;font-weight:700'
    if text.startswith('⚠'): return 'background:#fff3cd;color:#9a5600;font-weight:700'
    if 'NO DATA' in text: return 'background:#f1f3f5;color:#68727d;font-weight:700'
    return ''

device_ips=_load_device_ips(os.path.dirname(DEVICE_CONFIG))
if not overview_rows:
    print('No data loaded — nothing to summarize.')
else:
    import plotly.graph_objects as go
    overview_sections=[]
    for exp_id in sorted({r['Experiment'] for r in overview_rows}):
        exp_rows=[{**r,'IP':device_ips.get(r['Device'],'—')} for r in overview_rows if r['Experiment']==exp_id]
        index_rows=[]
        for device in sorted({r['Device'] for r in exp_rows}):
            dev_rows=[r for r in exp_rows if r['Device']==device]
            index_rows.append({'Device':device,'IP':dev_rows[0]['IP'],'Directory':dev_rows[0].get('Directory') or '—','Topics':', '.join(sorted({r['Topic'] for r in dev_rows}))})
        index_df=pd.DataFrame(index_rows)[INDEX_COLS].reset_index(drop=True)
        detail_df=pd.DataFrame(exp_rows)[DETAIL_COLS].sort_values(['Device','Topic']).reset_index(drop=True)
        index_html=index_df.style.set_caption('Device index').set_properties(**{'padding':'8px','text-align':'left'}).hide(axis='index').to_html()
        detail_html=(detail_df.style.map(_overview_status,subset=['Status']).set_caption('Topic details').set_properties(**{'padding':'8px','text-align':'right'}).set_properties(subset=['Device','Topic'],**{'text-align':'left'}).hide(axis='index').to_html())
        colors=['#159570' if str(status).startswith('✅') else '#e69f00' if str(status).startswith('⚠') else '#8b95a1' for status in detail_df['Status']]
        labels=[f'{device}<br>{topic}' for device,topic in zip(detail_df['Device'],detail_df['Topic'])]
        fig=go.Figure(go.Bar(x=labels,y=detail_df['Messages'],marker_color=colors,text=detail_df['Messages'],textposition='outside',hovertemplate='<b>%{x}</b><br>Messages: %{y}<extra></extra>'))
        fig.update_layout(title=f'Message coverage — {exp_id}',xaxis_title='Device / Topic',yaxis_title='Messages (count)',template='plotly_white',height=420,margin={'t':60,'b':110})
        figure_html=fig.to_html(full_html=False,include_plotlyjs=False)
        device_count=len({r['Device'] for r in exp_rows});topic_count=len({r['Topic'] for r in exp_rows});total_messages=sum(int(r['Messages']) for r in exp_rows);warning_count=sum(str(r.get('Status','')).startswith('⚠') or 'NO DATA' in str(r.get('Status','')) for r in exp_rows)
        cards=f'''<div style="display:grid;grid-template-columns:repeat(4,minmax(120px,1fr));gap:8px;margin:8px 0 16px"><div class="rdc-card"><small>Devices</small><strong>{device_count}</strong></div><div class="rdc-card"><small>Topics</small><strong>{topic_count}</strong></div><div class="rdc-card"><small>Total messages</small><strong>{total_messages}</strong></div><div class="rdc-card"><small>Checks needed</small><strong style="color:{'#c05a00' if warning_count else '#087f5b'}">{warning_count}</strong></div></div>'''
        body=f'<h3 style="overflow-wrap:anywhere">{escape(exp_id)}</h3>{cards}<h4>Devices</h4>{index_html}<h4>Topic details</h4>{detail_html}{figure_html}'
        overview_sections.append((exp_id,body))
    display(HTML(_filterable_experiment_html('rdc-overview-filter',overview_sections)))


## Section 3c — Message-Type Validity Checks

Device-independent structural checks with a valid/invalid coverage chart. IMU checks finite vectors, quaternion normalization, covariance shape, frame ID, and timestamp order. Camera checks dimensions, encoding, row step, payload size, frame ID, and timestamp order.

In [ ]:
print('=== Section 3c — Message-Type Validity Checks ===\n')
VALIDITY_TOPICS = {'/imu/data', '/camera/image_raw'}
validity_rows, validity_details = [], []

def header_stamp_ns(msg):
    stamp = getattr(getattr(msg, 'header', None), 'stamp', None)
    return None if stamp is None else int(stamp.sec) * 1_000_000_000 + int(stamp.nanosec)

def finite_vector(v):
    return bool(np.all(np.isfinite([v.x, v.y, v.z])))

if MODE == 'mock':
    print('Mock arrays contain no serialized ROS messages; structural checks are skipped.')
else:
    for ds in datasets:
        bag_dir = ds.get('bag_dir')
        wanted = sorted(VALIDITY_TOPICS.intersection(ds.get('topics', {})))
        if not bag_dir or not wanted:
            continue
        state = {t: {'messages': 0, 'invalid': 0, 'stamp_errors': 0, 'last': None} for t in wanted}
        for topic, msg, _ in read_ros2_bag_topics(bag_dir, wanted):
            item, reasons = state[topic], []
            item['messages'] += 1
            stamp = header_stamp_ns(msg)
            if stamp is None or stamp <= 0:
                reasons.append('missing/zero header stamp')
            elif item['last'] is not None and stamp < item['last']:
                item['stamp_errors'] += 1; reasons.append('header stamp moved backwards')
            if stamp is not None: item['last'] = stamp
            if not str(getattr(getattr(msg, 'header', None), 'frame_id', '')).strip():
                reasons.append('empty frame_id')
            if topic == '/imu/data':
                q = np.array([msg.orientation.x, msg.orientation.y, msg.orientation.z, msg.orientation.w])
                if not np.all(np.isfinite(q)): reasons.append('non-finite orientation')
                elif not 0.95 <= float(np.linalg.norm(q)) <= 1.05:
                    reasons.append(f'quaternion norm={np.linalg.norm(q):.3f}')
                if not finite_vector(msg.angular_velocity): reasons.append('non-finite angular velocity')
                if not finite_vector(msg.linear_acceleration): reasons.append('non-finite linear acceleration')
                for name in ('orientation_covariance','angular_velocity_covariance','linear_acceleration_covariance'):
                    if len(getattr(msg,name,[])) != 9: reasons.append(f'{name} length != 9')
            else:
                if msg.width <= 0 or msg.height <= 0: reasons.append('invalid width/height')
                if not str(msg.encoding).strip(): reasons.append('empty encoding')
                if msg.step <= 0: reasons.append('invalid row step')
                expected = int(msg.step) * int(msg.height)
                if expected <= 0 or len(msg.data) != expected:
                    reasons.append(f'payload={len(msg.data)}, expected={expected}')
            if reasons:
                item['invalid'] += 1
                if len(validity_details) < 20:
                    validity_details.append(f"{ds['label']} | {topic} | message {item['messages']}: " + '; '.join(reasons))
        exp_id, device_id = _experiment_and_device(ds['subdir'])
        for topic, item in state.items():
            validity_rows.append({'Experiment': exp_id, 'Device': device_id, 'Topic': topic,
                'Messages checked': item['messages'], 'Invalid messages': item['invalid'],
                'Header order errors': item['stamp_errors'],
                'Status': '✅ PASS' if item['messages'] and not item['invalid'] else '⚠️ CHECK'})
    if validity_rows:
        validity_df = pd.DataFrame(validity_rows).sort_values(['Experiment','Device','Topic']).reset_index(drop=True)
        def _validity_style(value):
            return ('background:#dcfce7;color:#087f5b;font-weight:700' if str(value).startswith('✅')
                    else 'background:#fff3cd;color:#9a5600;font-weight:700')
        display(validity_df.style.map(_validity_style,subset=['Status']).set_caption('IMU / Camera structural validity')
                .set_properties(**{'padding':'8px','text-align':'right'})
                .set_properties(subset=['Experiment','Device','Topic'], **{'text-align':'left'}).hide(axis='index'))
        from IPython.display import HTML
        import plotly.graph_objects as go
        valid_counts=validity_df['Messages checked']-validity_df['Invalid messages']
        labels=[f"{device}<br>{topic}" for device,topic in zip(validity_df['Device'],validity_df['Topic'])]
        fig=go.Figure([go.Bar(name='Valid',x=labels,y=valid_counts,marker_color='#159570'),
                       go.Bar(name='Invalid',x=labels,y=validity_df['Invalid messages'],marker_color='#d9485f')])
        fig.update_layout(barmode='stack',title='Message validity coverage',xaxis_title='Device / Topic',
                          yaxis_title='Messages checked (count)',template='plotly_white',height=390,margin={'b':100})
        display(HTML(fig.to_html(full_html=False,include_plotlyjs=False)))
        if validity_details:
            print('\nFirst validity issues (maximum 20):')
            for detail in validity_details: print(' -', detail)
    else:
        print('No IMU or Camera topics were loaded.')
